# 02 - Mosaic Sentinel-2 (merge tiles per year)

GEE automatically splits each year's Sentinel-2 export into several tiles once it gets large enough. This notebook merges those tiles back into a single raster per year and writes the result to `datasets/processed/sentinel2/` (`processed`, not `raw`, since this is already a derived output rather than the original download).

In [ ]:
# add scripts/download to sys.path so we can `from common import ...`
# (common.py is shared between the download and preprocessing notebooks, not duplicated)
import os, sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath("."))
if NOTEBOOK_DIR.name != "preprocessing":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "scripts" / "preprocessing"
os.chdir(NOTEBOOK_DIR)

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
DOWNLOAD_DIR = PROJECT_ROOT / "scripts" / "download"
if str(DOWNLOAD_DIR) not in sys.path:
    sys.path.insert(0, str(DOWNLOAD_DIR))

print("cwd:", os.getcwd())

## 1. Configuration

In [ ]:
from pathlib import Path

import rasterio
from rasterio.merge import merge

from common import RAW_DIR, PROCESSED_DIR, ensure_dir, get_logger

log = get_logger("mosaic_sentinel2")

SRC_DIR = RAW_DIR / "sentinel2gee"
OUT_DIR = PROCESSED_DIR / "sentinel2"
YEARS = range(2020, 2025)

# band order matches download_sentinel2.ipynb: B2,B3,B4,B8,B11,B12,NDVI,EVI,NDMI
BAND_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "EVI", "NDMI"]

## 2. Mosaic function

In [ ]:
def mosaic_year(year: int) -> Path:
    tiles = sorted(SRC_DIR.glob(f"sentinel2_banten_{year}-*.tif"))
    if not tiles:
        raise FileNotFoundError(f"No tiles found for {year} in {SRC_DIR}")

    log.info("%s: merging %d tiles", year, len(tiles))
    srcs = [rasterio.open(t) for t in tiles]
    try:
        mosaic, transform = merge(srcs)
        profile = srcs[0].profile.copy()
        profile.update(
            {
                "height": mosaic.shape[1],
                "width": mosaic.shape[2],
                "transform": transform,
                "compress": "deflate",
            }
        )
    finally:
        for s in srcs:
            s.close()

    ensure_dir(OUT_DIR)
    out_path = OUT_DIR / f"sentinel2_banten_{year}.tif"
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(mosaic)
        dst.descriptions = tuple(BAND_NAMES)

    size_mb = out_path.stat().st_size / 1024**2
    log.info("  -> %s (%.1f MB, shape=%s)", out_path.name, size_mb, mosaic.shape)
    return out_path

## 3. Run it for every year (2020-2024)

In [ ]:
results = {}
for year in YEARS:
    try:
        results[year] = mosaic_year(year)
    except Exception as e:  # noqa: BLE001
        log.warning("Failed to mosaic %s: %s", year, e)

print("\nDone. Files written:")
for year, path in results.items():
    print(f"  {year}: {path}")

## 4. Quick preview (true color & NDVI)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio

YEAR_TO_PREVIEW = max(results.keys()) if results else 2023
path = OUT_DIR / f"sentinel2_banten_{YEAR_TO_PREVIEW}.tif"

with rasterio.open(path) as src:
    red = src.read(3, masked=True)    # B4
    green = src.read(2, masked=True)  # B3
    blue = src.read(1, masked=True)   # B2
    ndvi = src.read(7, masked=True)   # NDVI

rgb = np.dstack([red.filled(0), green.filled(0), blue.filled(0)])
rgb = np.clip(rgb / 0.3, 0, 1)  # stretch reflectance (0-1) so it's visible

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(rgb)
axes[0].set_title(f"Sentinel-2 True Color {YEAR_TO_PREVIEW}")
axes[0].axis("off")

im = axes[1].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title(f"NDVI {YEAR_TO_PREVIEW}")
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.03)

plt.tight_layout()
plt.show()